# Tiny M1 utility prompt diagnostic — inference only

Why did the base model answer Unknown to all ten arithmetic questions? Compare the original instruction and an arithmetic-specific instruction on the pinned base and the already-saved tiny adapter: 40 answers, zero training updates. Original pilot scores/gates remain unchanged. No reasoning traces are requested.

Use the Kaggle GPU kernel and attach HF_TOKEN in Kaggle Secrets. **Commit/push the new diagnostic code/config before running:** the first cell clones main and stops if they are missing. The old training notebook does not need to be rerun.


In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect this notebook to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='utility-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/evaluation/utility_diagnostic.py').is_file(), 'New diagnostic files are absent on GitHub main. Commit/push them before running this notebook.'
assert Path('configs/evaluation/utility_prompt_diagnostic_v1.json').is_file()


Cloning into '/kaggle/working/utility-source-yo1zw6v_/repo'...


Repository commit: 5ec5960358f2a67704746dc09bf781a3c930f557


AssertionError: New diagnostic files are absent on GitHub main. Commit/push them before running this notebook.

In [ ]:
# Match the recorded tiny-run stack; clean the unused optional backend in Kaggle only.
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


In [ ]:
import json

plan_dir = Path(tempfile.mkdtemp(prefix='utility-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.evaluation.utility_diagnostic', '--config', 'configs/evaluation/utility_prompt_diagnostic_v1.json', '--output-dir', str(plan_dir), '--mode', 'plan'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_prediction_count'] == 40 and plan['training_updates'] == 0
print('Fixed prompts:', json.dumps(plan['system_prompts'], indent=2))


In [ ]:
# Read the private adapter with a secret passed only through the child environment.
from kaggle_secrets import UserSecretsClient

evaluation_env = os.environ.copy()
evaluation_env['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
output_dir = Path(tempfile.mkdtemp(prefix='m1-utility-prompt-', dir=remote_root))
print('Evaluation outputs:', output_dir, flush=True)
try:
    subprocess.run([sys.executable, '-u', '-m', 'src.evaluation.utility_diagnostic', '--config', 'configs/evaluation/utility_prompt_diagnostic_v1.json', '--output-dir', str(output_dir), '--mode', 'evaluate'], env=evaluation_env, check=True)
finally:
    evaluation_env.pop('HF_TOKEN', None)


In [ ]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete'] and summary['training_updates'] == 0
print('Original outputs reproduced:', summary['original_v2_reproduction'])
print('Paired comparison ready:', summary['paired_comparison_ready'])
for file in sorted(output_dir.glob('*.jsonl')):
    print('\n' + file.name)
    for line in file.read_text().splitlines():
        row = json.loads(line)
        print(row['example_id'], '|', row['question'], '| expected:', row['answer'], '| got:', repr(row['raw_response']))
print('Original tiny pilot remains failed. Review these outputs before choosing another training run.')


In [ ]:
# Export reports only. The restored adapter stays outside the small ZIP.
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
